In [1]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

import MDAnalysis as mda
from MDAnalysis.analysis import distances
import networkx as nx

from polym_analysis import *
from gromacs_topology import *

/jet/home/schwinns/pkgs/my_scripts/polym_analysis.py:1976: SyntaxWarning: invalid escape sequence '\A'
  ax.set_xlabel('z dimension ($\AA$)')
/jet/home/schwinns/pkgs/my_scripts/polym_analysis.py:1977: SyntaxWarning: invalid escape sequence '\A'
  ax.set_ylabel(ydim+' dimension ($\AA$)')
/jet/home/schwinns/.conda/envs/mda/lib/python3.12/site-packages/Bio/Application/__init__.py:40: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
# CPK color scheme
cpk_colors = {
    'H': '#FFFFFF', 'C': '#000000', 'N': '#3050F8', 'O': '#FF0D0D',
    'F': '#90E050', 'P': '#FF8000', 'S': '#FFFF30', 'Cl': '#1FF01F',
    'Br': '#A62929', 'I': '#940094', 'Na': '#AB5CF2', 'Mg': '#8AFF00',
    'Ca': '#3DFF00', 'Fe': '#E06633', 'Zn': '#7D80B0'
}

def draw_graph(G: nx.Graph, pos=None, show_atoms=True):
    '''
    Draw a NetworkX graph with nodes colored by element type.

    Parameters
    ----------
    G: NetworkX Graph
        The graph to draw.
    pos: dict, optional
        A dictionary mapping node indices to positions. If None, a spring layout will be used.
    '''

    # Extract node colors and labels
    node_colors = [G.nodes[node]['color'] for node in G.nodes()]
    node_sizes = [75*G.nodes[node]['atom'].mass for node in G.nodes()]

    # Create the plot
    plt.figure(figsize=(8, 6))
    if pos is None:
        pos = nx.spring_layout(G, k=1, iterations=50)

    # Draw edges first
    nx.draw_networkx_edges(G, pos, alpha=0.6, width=2)

    # Draw nodes with colors
    if show_atoms:
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, edgecolors='k', 
                            node_size=node_sizes, alpha=0.9)

    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [3]:
def random_amides(polym, num, seed=12345):
    '''Randomly select num of amide bonds to break'''

    sel = polym.universe.select_atoms('type n')
    n = len(sel)
    rng = np.random.default_rng(seed=seed)
    rand_idx = rng.choice(np.arange(0,n), size=num, replace=False)
    
    return sel[rand_idx]


def get_bond(top, atomtype1, atomtype2):
    '''Get the bond between two atom types in a topology.'''

    atomtypes = [atomtype1, atomtype2]

    for b in top.bonds:
        my_types = [a.type for a in b.atoms]
        if Counter(my_types) == Counter(atomtypes):
            return b
    return None


def get_angle(top, atomtype1, atomtype2, atomtype3):
    '''Get the angle among three atom types in a topology.'''

    atomtypes = [atomtype1, atomtype2, atomtype3]

    for a in top.angles:
        my_types = [a.type for a in a.atoms]
        if Counter(my_types) == Counter(atomtypes):
            return a
    return None


def get_dihedral(top, atomtype1, atomtype2, atomtype3, atomtype4, is_improper=False):
    '''Get the dihedral among four atom types in a topology. Should specify if this dihedral is improper.'''

    atomtypes = [atomtype1, atomtype2, atomtype3, atomtype4]
    improper_types = [2, 4] # either harmonic (2) or periodic (4) improper

    for d in top.dihedrals:
        my_types = [a.type for a in d.atoms]
        if Counter(my_types) == Counter(atomtypes):
            if is_improper and d.type in improper_types:
                return d
            
            if not is_improper and d.type not in improper_types:
                return d
            
    return None

## Break bonds and determine new residues

In [10]:
path = '/ocean/projects/chm230020p/schwinns/Projects/polymer_membrane/REACTER/Lx5_Ly5_Lz10/chlorination/50pct_deprotonated/NaCl_0.2M/hydrolyzed_chlorination/random_10pct/build_chlorinated/'

xlink_c = 'c and bonded type n'
xlink_n = 'n'
term_n = 'nh'
cl_type = 'cl'
oh_type = 'oh'
ow_type = 'OW'
hw_type = 'HW'
cation_type = 'Na'
anion_type = 'Cl'

poly = PolymAnalysis(path+'removed_waters.gro', frmt='GRO', tpr_file=path+'removed_waters.tpr',
                    xlink_c=xlink_c, xlink_n=xlink_n, term_n=term_n,
                    cl_type=cl_type, oh_type=oh_type, ow_type=ow_type, hw_type=hw_type,
                    cation_type=cation_type, anion_type=anion_type)

Initializing polyamide membrane for analysis...
PA membrane consists of 716 MPD monomers, 444 TMC monomers, 5914 waters for a total of 35468 atoms
Finished loading PA membrane for analysis!



In [11]:
# determine crosslinks to break, waters to remove, and positions to add Cl and OH
target_crosslink_pct = 0.8
radius_for_water_search = 6  # angstroms
rng_seed = 3

n_xlink = len(poly.universe.select_atoms('type n'))
n_possible = len(poly.universe.select_atoms('type n nh'))
n_to_break = n_xlink - int(target_crosslink_pct * n_possible)
original_PA_molecules = poly.polymer.n_residues

poly.create_polymer_graphs()
n_components = [nx.number_connected_components(g) for g in poly.polymer_graphs]

print(f'Going from {100 * n_xlink / n_possible:.1f}% crosslinking to {100 * (n_xlink - n_to_break) / n_possible:.1f}% crosslinking by breaking {n_to_break} amide bonds')
                                  
print('\nNumber of bonds before breaking crosslinks:', len(poly.universe.bonds))
print('Number of polymer components before breaking crosslinks:', sum(n_components))

# delete amide bonds
amide_Ns = random_amides(poly, n_to_break, seed=rng_seed)
amide_Cs = poly.universe.select_atoms('(type c) and (bonded group amide_Ns)', amide_Ns=amide_Ns)
to_break = [b for b in amide_Ns.bonds if 'n' in b.type and 'c' in b.type]

print(f'Breaking {len(to_break)} bonds')
poly.universe.delete_bonds(to_break)

poly.polymer = poly.universe.select_atoms(f'not type {poly.ow_type} {poly.hw_type} {poly.anion_type} {poly.cation_type}') # reselect polymer atoms after deleting waters
poly.create_polymer_graphs()
n_components = [nx.number_connected_components(g) for g in poly.polymer_graphs]

print('\nNumber of bonds after breaking crosslinks:', len(poly.universe.bonds))
print('Number of polymer chains after breaking crosslinks:', sum(n_components))

Going from 93.2% crosslinking to 80.0% crosslinking by breaking 189 amide bonds

Number of bonds before breaking crosslinks: 30799
Number of polymer components before breaking crosslinks: 14
Breaking 189 bonds

Number of bonds after breaking crosslinks: 30610
Number of polymer chains after breaking crosslinks: 117


In [12]:
# Reassign residue indices based on connected components
atom_to_new_resindex = {}
new_resindex_to_atom = {}
current_resid = 1
n_new_residues = 0

for residue in poly.polymer.residues:
    res_atoms = residue.atoms
    res_graph = build_graph_from_atoms(res_atoms)
    
    # Get connected components for this residue
    components = list(nx.connected_components(res_graph))
    
    for component in components:
        new_resindex_to_atom[n_new_residues] = []
        # Assign all atoms in this component to the same new residue
        for atom_idx in component:
            atom_to_new_resindex[atom_idx] = n_new_residues
            new_resindex_to_atom[n_new_residues].append(atom_idx)
        
        new_resindex_to_atom[n_new_residues] = np.array(new_resindex_to_atom[n_new_residues])
        n_new_residues += 1

# Create a new empty universe with correct number of residues
n_atoms = len(poly.polymer.atoms)
new_polymer_u = mda.Universe.empty(
    n_atoms=n_atoms,
    n_residues=n_new_residues,
    atom_resindex=[atom_to_new_resindex[atom.index] for atom in poly.polymer.atoms],
    trajectory=True
)

# Copy over topology attributes
new_polymer_u.add_TopologyAttr('name', [atom.name for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('type', [atom.type for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('element', [atom.element for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('mass', [atom.mass for atom in poly.polymer.atoms])
new_polymer_u.add_TopologyAttr('resname', [f'PA{i}' for i in range(1, n_new_residues + 1)])
new_polymer_u.add_TopologyAttr('resid', [i for i in range(1, n_new_residues + 1)])

# Copy positions and bonds
new_polymer_u.atoms.positions = poly.polymer.atoms.positions
bond_indices = [(b.atoms[0].index, b.atoms[1].index) for b in poly.polymer.bonds]
new_polymer_u.add_TopologyAttr('bonds', bond_indices)

# Merge with non-polymer atoms
non_polymer = poly.universe.atoms - poly.polymer.atoms
merged_u = mda.Merge(new_polymer_u.atoms, non_polymer)
merged_u.dimensions = poly.universe.dimensions

# Reassign resids to be sequential
# Polymer residues: 1 to n_new_residues
# Non-polymer residues: n_new_residues+1 onwards
n_polymer_atoms = len(new_polymer_u.atoms)
non_polymer_start_idx = n_new_residues + 1

# Get the non-polymer residues and renumber them
for i, res in enumerate(merged_u.residues[n_new_residues:], start=non_polymer_start_idx):
    for atom in res.atoms:
        atom.residue.resid = i

# Replace the universe
poly.universe = merged_u

print(f'Number of new polymer residues: {n_new_residues}')
print(f'Number of polymer chains after breaking crosslinks: {sum(n_components)}')

poly.universe.atoms.write('broken_bonds.gro')

Number of new polymer residues: 117
Number of polymer chains after breaking crosslinks: 117


In [13]:
# update the topology file
# my GromacsTopology class writes out only using the molecules, so it should be fine if we update all the molecule information

top = GromacsTopology('./removed_waters.top', verbose=False)
polymer_idx = poly.universe.select_atoms(f'not type {poly.ow_type} {poly.hw_type} {poly.anion_type} {poly.cation_type}').indices
broken_amide_N_idx = amide_Ns.indices
broken_amide_C_idx = amide_Cs.indices

# initialize new molecules
new_molecules = []
for i in range(1, n_new_residues + 1):
    mol = Molecule(f'PA{i}', 1)
    setattr(mol, 'atoms', [])
    setattr(mol, 'bonds', [])
    setattr(mol, 'angles', [])
    setattr(mol, 'dihedrals', [])

    mol.directives = ['bonds', 'angles', 'dihedrals']
    new_molecules.append(mol)

# loop through all atoms and assign to new molecules
for atom in top.atoms[polymer_idx]:
    # add to new molecule
    resindex = atom_to_new_resindex[atom.idx]
    my_map = {orig_idx : new_idx+1 for new_idx,orig_idx in enumerate(new_resindex_to_atom[resindex])} # renumbering map within new residue
    
    # update atom attributes for new molecule
    atom.num = my_map[atom.idx]
    atom.resnum = resindex + 1
    atom.resname = f'PA{resindex + 1}'

    my_mol = new_molecules[resindex]
    my_mol.atoms.append(atom)

# sort the atoms in each new molecule by their new atom number
for mol in new_molecules:
    mol.atoms.sort(key=lambda a: a.num)

# loop through all bonds and assign to new molecules
n_bonds_skipped = 0
for bond in top.bonds: # this is all polymer bonds because water uses settles and monoatomic ions have no bonds
    atom1, atom2 = bond.atoms
    my_types = [a.type for a in bond.atoms]
    my_mol = new_molecules[atom_to_new_resindex[atom1.idx]]

    if 'c' in my_types and 'n' in my_types:
        c_atom = atom1 if atom1.type == 'c' else atom2
        n_atom = atom1 if atom1.type == 'n' else atom2
        if c_atom.idx in broken_amide_C_idx and n_atom.idx in broken_amide_N_idx:
            # skip amide bonds that have been broken, which deletes them from the new molecule
            # print(f'Skipping bond between atoms {atom1.idx}:{atom1.type} and {atom2.idx}:{atom2.type}')
            n_bonds_skipped += 1
            continue

    my_mol.bonds.append(bond)

print(f'Skipped {n_bonds_skipped} bonds corresponding to broken amide bonds')

# loop through all angles and assign to new molecules
n_angles_skipped = 0
for angle in top.angles: # this is all polymer angles because water uses settles and monoatomic ions have no angles
    atom1, atom2, atom3 = angle.atoms
    my_types = [a.type for a in angle.atoms]
    my_mol = new_molecules[atom_to_new_resindex[atom1.idx]]

    if 'c' in my_types and 'n' in my_types:
        c_atom = [atom for atom in angle.atoms if atom.type == 'c'][0]
        n_atom = [atom for atom in angle.atoms if atom.type == 'n'][0]
        if c_atom.idx in broken_amide_C_idx and n_atom.idx in broken_amide_N_idx:
            # skip angles that involve broken amide bonds
            n_angles_skipped += 1
        continue

    my_mol.angles.append(angle)

print(f'Skipped {n_angles_skipped} angles corresponding to broken amide bonds')

# loop through all dihedrals and assign to new molecules
n_dihedrals_skipped = 0
for dihedral in top.dihedrals: # this is all polymer dihedrals because water uses settles and monoatomic ions have no dihedrals
    atom1, atom2, atom3, atom4 = dihedral.atoms
    my_types = [a.type for a in dihedral.atoms]
    my_mol = new_molecules[atom_to_new_resindex[atom1.idx]]

    if 'c' in my_types and 'n' in my_types:
        c_atom = [atom for atom in dihedral.atoms if atom.type == 'c'][0]
        n_atom = [atom for atom in dihedral.atoms if atom.type == 'n'][0]
        if c_atom.idx in broken_amide_C_idx and n_atom.idx in broken_amide_N_idx:
            # skip dihedrals that involve broken amide bonds
            n_dihedrals_skipped += 1
        continue

    my_mol.dihedrals.append(dihedral)

print(f'Skipped {n_dihedrals_skipped} dihedrals corresponding to broken amide bonds')

# add the water and ion molecules back
for mol in top.molecules:
    if not mol.name.startswith('PA'):
        new_molecules.append(mol)

# update topology molecules
top.molecules = new_molecules
top.write('broken_bonds.top')



18 molecule types in the topology for a total of 6017 molecules
Skipped 189 bonds corresponding to broken amide bonds
Skipped 756 angles corresponding to broken amide bonds
Skipped 1887 dihedrals corresponding to broken amide bonds
Finished writing topology to broken_bonds.top!


In [15]:
# reassign global atom indices to the GRO file
u = mda.Universe('broken_bonds.gro')
global_map = {}
new_idx = 0

# start with polymer atoms
for mol in top.molecules[:-4]:  # last 4 molecules are non-polymer
    for atom in mol.atoms:
        global_map[atom.idx] = new_idx
        new_idx += 1

# # start with polymer atoms
# n_polymer_mols = len(top.molecules) - 4  # last 4 molecules are non-polymer
# for i in range(n_polymer_mols):
#     polymer_res = u.select_atoms(f'resname PA{i+1}')
#     for atom in polymer_res:
#         global_map[atom.ix] = new_idx
#         new_idx += 1

print(f'Next available global atom index after polymer atoms: {new_idx}')

# then non-polymer atoms (first, waters)
for atom in u.select_atoms('resname SOL'):
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after water atoms: {new_idx}')

n_mols_NA_1 = top.molecules[-3].n_mols # NA molecules are split in topology, this is first set
for atom in u.select_atoms('resname NA')[:n_mols_NA_1]:
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after 1st Na atoms: {new_idx}')

# then Cl ions
n_mols_CL = top.molecules[-2].n_mols
for atom in u.select_atoms('resname CL'):
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after Cl atoms: {new_idx}')

for atom in u.select_atoms('resname NA')[n_mols_NA_1:]:
    global_map[atom.ix] = new_idx
    new_idx += 1

print(f'Next available global atom index after 2nd Na atoms: {new_idx}')

u.add_TopologyAttr('ids', [global_map[atom.ix] for atom in u.atoms])
u.atoms.sort('ids').write('broken_bonds_sorted.gro')

Next available global atom index after polymer atoms: 17637
Next available global atom index after water atoms: 35379
Next available global atom index after 1st Na atoms: 35391
Next available global atom index after Cl atoms: 35403
Next available global atom index after 2nd Na atoms: 35468
